In [ ]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("RUSTFS_ENDPOINT", "https://hpi-s3.mittelstand-zukunftslab.de"),
    aws_access_key_id=os.getenv("RUSTFS_ACCESS_KEY", "rustfsadmin"),
    aws_secret_access_key=os.getenv("RUSTFS_SECRET_KEY", ""),
    region_name="us-east-1",
)

response = s3.list_buckets()
print([b["Name"] for b in response["Buckets"]])

In [ ]:
import lancedb

db = lancedb.connect(
    "s3://deepfix-index/legal_rag_bench",
    storage_options={
        "endpoint": os.getenv("RUSTFS_ENDPOINT", "https://hpi-s3.mittelstand-zukunftslab.de"),
        "aws_access_key_id": os.getenv("RUSTFS_ACCESS_KEY", "rustfsadmin"),
        "aws_secret_access_key": os.getenv("RUSTFS_SECRET_KEY", ""),
        "aws_region": "us-east-1",
        "allow_http": "true",
        "aws_virtual_hosted_style_request": "false",
    },
)
print(db.list_tables() if hasattr(db, "list_tables") else db.table_names())

In [ ]:
LITELLM_URL = os.getenv("LITELLM_URL", "https://litellm.mittelstand-zukunftslab.de")
LITELLM_KEY = os.getenv("LITELLM_KEY", "")

In [ ]:
from datasets import load_dataset
qa_dataset = load_dataset("isaacus/legal-rag-bench", "qa", split="test")
print(len(qa_dataset))

In [ ]:
from datasets import load_dataset

corpus_dataset = load_dataset("isaacus/legal-rag-bench", "corpus", split="test")
print(len(corpus_dataset))

In [ ]:
import litellm
import pandas as pd

def embed_batch(texts, batch_size=10):
    all_vectors = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        resp = litellm.embedding(
            model="openai/qwen/qwen3-embedding-8b",
            input=chunk,
            api_key=LITELLM_KEY,
            api_base=LITELLM_URL,
            timeout=60
        )
        all_vectors.extend([d["embedding"] for d in resp.data])
    return all_vectors

test_rows = corpus_dataset.select(range(20))
test_texts = [r["text"] for r in test_rows]
test_vectors = embed_batch(test_texts)

test_data = [
    {"id": row["id"], "title": row["title"], "text": row["text"], "vector": vec}
    for row, vec in zip(test_rows, test_vectors)
]

test_df = pd.DataFrame(test_data)
table = db.create_table("legal_passages_test", data=test_df, mode="overwrite")
print(table.count_rows())

In [ ]:
query_text = "jury excuse bias"
query_vector = embed_batch([query_text])[0]

results = (
    table.search(query_type="hybrid")
    .vector(query_vector)
    .text(query_text)
    .limit(3)
    .to_pandas()
)
print(results[["id", "title", "text"]])

In [ ]:
full_texts = [r["text"] for r in corpus_dataset]
print(f"Embedding {len(full_texts)} passages... this will take a while")

full_vectors = embed_batch(full_texts, batch_size=10)
print(f"Done embedding: {len(full_vectors)} vectors")

In [ ]:
full_data = [
    {"id": row["id"], "title": row["title"], "text": row["text"], "vector": vec}
    for row, vec in zip(corpus_dataset, full_vectors)
]

full_df = pd.DataFrame(full_data)
full_table = db.create_table("legal_passages", data=full_df, mode="overwrite")
print(full_table.count_rows())

In [ ]:
from lancedb.index import FTS
full_table.create_index("text", config=FTS())

In [ ]:
def hybrid_answer_question_v2(question, top_k=10):
    q_vector = embed_batch([question])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(question)
        .limit(top_k)
        .to_pandas()
    )
    context_chunks = list(results["text"])
    context_str = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context_str}

Question: {question}

Answer:"""

    chat_response = litellm.completion(
        model="qwen3-8-27b",
        messages=[{"role": "user", "content": prompt}],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=60
    )
    return {
        "context": context_chunks,
        "answer": chat_response.choices[0].message.content
    }

In [ ]:
def diagnose_lancedb(qa_index, limit=20):
    qa = qa_dataset[qa_index]
    question = qa["question"]
    correct_id = qa["relevant_passage_id"]

    fts_results = full_table.search(question, query_type="fts").limit(limit).to_pandas()
    fts_ids = list(fts_results["id"])

    q_vector = embed_batch([question])[0]
    vec_results = full_table.search(q_vector).limit(limit).to_pandas()
    vec_ids = list(vec_results["id"])

    print(f"Q: {question[:70]}...")
    print(f"Correct ID: {correct_id}")
    print(f"FTS-only top {limit}: found={correct_id in fts_ids} | ids={fts_ids}")
    print(f"Vector-only top {limit}: found={correct_id in vec_ids} | ids={vec_ids}")
    print("---")

for i in [11, 28, 21]:
    diagnose_lancedb(i)

In [ ]:
def test_hybrid_wider(qa_index, limit=10):
    qa = qa_dataset[qa_index]
    question = qa["question"]
    correct_id = qa["relevant_passage_id"]
    q_vector = embed_batch([question])[0]

    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(question)
        .limit(limit)
        .to_pandas()
    )
    ids = list(results["id"])
    print(f"Q: {question[:70]}...")
    print(f"Correct ID: {correct_id} | Found: {correct_id in ids} | Rank: {ids.index(correct_id)+1 if correct_id in ids else 'N/A'}")
    print(f"All retrieved: {ids}")
    print("---")

for i in [11, 28, 21]:
    test_hybrid_wider(i, limit=10)

In [ ]:
from deepeval.test_case import LLMTestCase
import time

test_cases_v2 = []
failed_v2 = []

for qa in qa_dataset:
    try:
        r = hybrid_answer_question_v2(qa["question"])
        test_cases_v2.append(LLMTestCase(
            input=qa["question"],
            actual_output=r["answer"],
            expected_output=qa["answer"],
            retrieval_context=r["context"],
        ))
    except Exception as e:
        failed_v2.append({"id": qa.get("id"), "error": str(e)})
    time.sleep(0.5)

print(f"Succeeded: {len(test_cases_v2)} / {len(qa_dataset)}")
print(f"Failed: {len(failed_v2)}")

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from deepeval.models.base_model import DeepEvalBaseLLM

class ProxyJudgeLLM(DeepEvalBaseLLM):
    def __init__(self, model, api_base, api_key):
        self.model, self.api_base, self.api_key = model, api_base, api_key
    def load_model(self):
        return self.model
    def generate(self, prompt: str) -> str:
        r = litellm.completion(model=self.model, messages=[{"role":"user","content":prompt}],
                                api_key=self.api_key, api_base=self.api_base, timeout=60)
        return r.choices[0].message.content
    async def a_generate(self, prompt: str) -> str:
        r = await litellm.acompletion(model=self.model, messages=[{"role":"user","content":prompt}],
                                       api_key=self.api_key, api_base=self.api_base, timeout=60)
        return r.choices[0].message.content
    def get_model_name(self):
        return self.model

judge = ProxyJudgeLLM("openai/openrouter/openai/gpt-4.1-mini", LITELLM_URL, LITELLM_KEY)
print(judge.generate("Say OK"))

In [ ]:
from deepeval.metrics import FaithfulnessMetric, ContextualPrecisionMetric, GEval
from deepeval.test_case import LLMTestCaseParams

faithfulness = FaithfulnessMetric(threshold=0.80, model=judge, async_mode=False)
contextual_precision = ContextualPrecisionMetric(threshold=0.75, model=judge, async_mode=False)
answer_correctness = GEval(
    name="Answer Correctness",
    criteria="Determine whether the actual output is factually correct and complete relative to the expected output, without omitting key legal conclusions.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.75,
    model=judge,
    async_mode=False,
)

In [ ]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig

eval_results_v2 = evaluate(
    test_cases_v2,
    metrics=[faithfulness, contextual_precision, answer_correctness],
    async_config=AsyncConfig(run_async=False)
)

In [ ]:
import pandas as pd

metrics_list = [("Faithfulness", faithfulness), ("Contextual Precision", contextual_precision), ("Answer Correctness", answer_correctness)]

results_v2 = []
errors_v2 = []

for idx, tc in enumerate(test_cases_v2):
    row = {"input": tc.input, "actual_output": tc.actual_output}
    for name, metric in metrics_list:
        success = False
        for attempt in range(3):
            try:
                metric.measure(tc)
                row[f"{name}_score"] = metric.score
                row[f"{name}_reason"] = metric.reason
                success = True
                break
            except Exception as e:
                last_error = str(e)
        if not success:
            row[f"{name}_score"] = None
            row[f"{name}_reason"] = f"FAILED after 3 attempts: {last_error}"
            errors_v2.append({"index": idx, "metric": name, "error": last_error})

    results_v2.append(row)

    if idx % 5 == 0:
        pd.DataFrame(results_v2).to_csv("deepeval_results_lancedb_v2_partial.csv", index=False)
        print(f"Checkpoint saved at question {idx+1}/{len(test_cases_v2)}")

pd.DataFrame(results_v2).to_csv("deepeval_results_lancedb_v2_final.csv", index=False)
print(f"Done. Total errors: {len(errors_v2)}")

In [ ]:
df_v2 = pd.DataFrame(results_v2)

summary_v2 = df_v2[["Faithfulness_score", "Contextual Precision_score", "Answer Correctness_score"]].mean()
print(summary_v2)

print("\nPer-metric pass rate (score >= threshold):")
print(f"Faithfulness >= 0.80: {(df_v2['Faithfulness_score'] >= 0.80).mean():.2%}")
print(f"Contextual Precision >= 0.75: {(df_v2['Contextual Precision_score'] >= 0.75).mean():.2%}")
print(f"Answer Correctness >= 0.75: {(df_v2['Answer Correctness_score'] >= 0.75).mean():.2%}")

In [ ]:
worst_precision = df_v2.sort_values("Contextual Precision_score").head(5)
for _, r in worst_precision.iterrows():
    print(r["input"])
    print("Score:", r["Contextual Precision_score"], "| Reason:", r["Contextual Precision_reason"])
    print("---")

In [ ]:
check_ids = [
    "Harry is serving as a juror",
    "Does Victoria have a law",
    "One of the elements of aggravated carjacking",
    "At dinner, Kim and Frank",
    "Two friends agree to race"
]
for c in check_ids:
    match = df_v2[df_v2["input"].str.contains(c[:20], regex=False)]
    print(match[["input", "Faithfulness_score", "Answer Correctness_score"]].to_string())
    print("---")

In [ ]:
low_prec = df_v2[df_v2["Contextual Precision_score"] < 0.2]
high_prec = df_v2[df_v2["Contextual Precision_score"] >= 0.75]

print(f"Zero/near-zero precision (n={len(low_prec)}):")
print(low_prec[["Faithfulness_score", "Answer Correctness_score"]].mean())
print()
print(f"High precision (n={len(high_prec)}):\n")
print(high_prec[["Faithfulness_score", "Answer Correctness_score"]].mean())

In [ ]:
def rewrite_query_for_retrieval(question, retries=3, timeout=60):
    prompt = f"""Read this legal question. In one short phrase, name the specific legal doctrine, rule, or concept it is testing (not the case facts).\n\nQuestion: {question}\n\nLegal concept:"""
    for attempt in range(retries):
        try:
            r = litellm.completion(model="qwen3-8-27b",
                                    messages=[{"role":"user","content":prompt}],
                                    api_key=LITELLM_KEY, api_base=LITELLM_URL, timeout=timeout)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retries - 1:
                return f"FAILED: {e}"
            time.sleep(3)

In [ ]:
failing_questions = low_prec["input"].tolist()
rewritten_concepts = []
for q in failing_questions:
    rewritten = rewrite_query_for_retrieval(q)
    rewritten_concepts.append(rewritten)
    print("ORIGINAL:", q[:80])
    print("REWRITTEN CONCEPT:", rewritten)
    print("---")

In [ ]:
def test_rewritten_retrieval(original_q, rewritten_q):
    q_vector = embed_batch([rewritten_q])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(rewritten_q)
        .limit(10)
        .to_pandas()
    )
    print("ORIGINAL:", original_q[:80])
    print("REWRITTEN:", rewritten_q)
    print("TOP 3 RETRIEVED CHUNKS:")
    for t in results["text"].head(3):
        print(" -", t[:150])
    print("---")

for q, r in zip(failing_questions, rewritten_concepts):
    test_rewritten_retrieval(q, r)

In [ ]:
def hybrid_answer_question_v3(question, top_k=10):
    concept = rewrite_query_for_retrieval(question)
    combined_query = f"{question}\n\nRelevant legal concept: {concept}"

    q_vector = embed_batch([combined_query])[0]
    results = (
        full_table.search(query_type="hybrid")
        .vector(q_vector)
        .text(combined_query)
        .limit(top_k)
        .to_pandas()
    )
    context_chunks = list(results["text"])
    context_str = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.\n\nContext:\n{context_str}\n\nQuestion: {question}\n\nAnswer:"""

    chat_response = litellm.completion(
        model="qwen3-8-27b",
        messages=[{"role": "user", "content": prompt}],
        api_key=LITELLM_KEY,
        api_base=LITELLM_URL,
        timeout=60
    )
    return {"context": context_chunks, "answer": chat_response.choices[0].message.content}

In [ ]:
failing_ids = low_prec.index.tolist()
test_cases_v3 = []

for idx in failing_ids:
    qa = qa_dataset[idx]
    r = hybrid_answer_question_v3(qa["question"])
    test_cases_v3.append(LLMTestCase(
        input=qa["question"], actual_output=r["answer"],
        expected_output=qa["answer"], retrieval_context=r["context"]
    ))

results_v3 = []
for tc in test_cases_v3:
    row = {"input": tc.input}
    for name, metric in metrics_list:
        try:
            metric.measure(tc)
            row[f"{name}_score"] = metric.score
        except Exception as e:
            row[f"{name}_score"] = None
    results_v3.append(row)

df_v3 = pd.DataFrame(results_v3)
print(df_v3.mean(numeric_only=True))

In [ ]:
import pandas as pd

df_v2_final = pd.DataFrame(results_v2)

comparison = pd.DataFrame({
    "system": ["Qdrant_baseline", "LanceDB_hybrid_v2"],
    "faithfulness": [0.877, df_v2_final["Faithfulness_score"].mean()],
    "contextual_precision": [0.71, df_v2_final["Contextual Precision_score"].mean()],
    "answer_correctness": [0.593, df_v2_final["Answer Correctness_score"].mean()],
    "n_questions": [100, 100]
})

df_v2_final.to_csv("deepeval_lancedb_v2_per_question.csv", index=False)
comparison.to_csv("deepeval_lancedb_vs_qdrant_summary.csv", index=False)
print(comparison)
